<a href="https://colab.research.google.com/github/quintussse/credit_scoring_ai_project/blob/main/projet_credit_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scoring de crédit équitable

## Étape 2 — Récupérer les données

In [ ]:
# On télécharge les données et on les ouvre dans un tableau
import pandas as pd
url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
data = pd.read_csv(url)
print("Nombre de personnes dans le fichier :", len(data))
print("Noms des colonnes :")
print(list(data.columns))
data.head()

Nombre de personnes dans le fichier : 1000
Noms des colonnes :
['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'credit_risk']


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee/official,1,no,yes,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,no,yes,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement/life insurance,45,none,for free,1,skilled employee/official,2,no,yes,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown/no property,53,none,for free,2,skilled employee/official,2,no,yes,0


## Étape 3.1 — On regarde la cible

In [ ]:
# La colonne "credit_risk" indique si la personne est bon ou mauvais payeur
print(data["credit_risk"].value_counts())

credit_risk
1    700
0    300
Name: count, dtype: int64


## Étape 3.2 — On prépare les données

In [ ]:
# On travaille sur une copie, pour garder "data" intact
df = data.copy()
# IMPORTANT : credit_risk contient DEJA des nombres (1 = bon payeur, 0 = mauvais).
# On NE compare donc PAS a "good" : on convertit simplement en entier.
df["cible"] = df["credit_risk"].astype(int)
df = df.drop(columns=["credit_risk"])
# Verification immediate : on doit voir 1 -> 700 et 0 -> 300
print("Verification de la cible :")
print(df["cible"].value_counts())
# On transforme les colonnes texte en colonnes chiffres
df = pd.get_dummies(df, drop_first=True)
print("Donnees pretes. Nombre de colonnes :", df.shape[1])
df.head()

Verification de la cible :
cible
1    700
0    300
Name: count, dtype: int64
Donnees pretes. Nombre de colonnes : 49


,duration,amount,installment_rate,present_residence,age,number_credits,people_liable,cible,status_... >= 200 DM / salary for at least 1 year,status_0 <= ... < 200 DM,...,property_unknown/no property,other_installment_plans_none,other_installment_plans_stores,housing_own,housing_rent,job_skilled employee/official,job_unemployed/unskilled - non-resident,job_unskilled - resident,telephone_yes,foreign_worker_yes
0,6,1169,4,4,67,2,1,1,False,False,...,False,True,False,True,False,True,False,False,True,True
1,48,5951,2,2,22,1,1,0,False,True,...,False,True,False,True,False,True,False,False,False,True
2,12,2096,2,3,49,1,2,1,False,False,...,False,True,False,True,False,False,False,True,False,True
3,42,7882,2,4,45,1,2,1,False,False,...,False,True,False,False,False,True,False,False,False,True
4,24,4870,3,4,53,2,2,0,False,False,...,True,True,False,False,False,True,False,False,False,True


## Étape 4.1 — On sépare apprentissage / test (avec `stratify=y`)

In [ ]:
from sklearn.model_selection import train_test_split
# X = toutes les infos ; y = la reponse (1 = bon payeur, 0 = mauvais payeur)
X = df.drop(columns=["cible"])
y = df["cible"]
# 80 % pour apprendre, 20 % pour tester.
# stratify=y garantit la MEME proportion de bons/mauvais payeurs des deux cotes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Exemples pour apprendre :", len(X_train))
print("Exemples pour tester :", len(X_test))
print("Dans y_train -> 1 :", (y_train == 1).sum(), "| 0 :", (y_train == 0).sum())
print("Dans y_test  -> 1 :", (y_test == 1).sum(), "| 0 :", (y_test == 0).sum())

Exemples pour apprendre : 800
Exemples pour tester : 200
Dans y_train -> 1 : 560 | 0 : 240
Dans y_test  -> 1 : 140 | 0 : 60


## Étape 4.2 — On entraîne le modèle et on le note

*   List item
*   List item



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
# On cree le modele et on le fait apprendre sur le groupe d'entrainement
modele = LogisticRegression(max_iter=5000)
modele.fit(X_train, y_train)
# On lui demande de deviner sur les exemples de test (qu'il n'a jamais vus)
proba = modele.predict_proba(X_test)[:, 1]
# Note de qualite : 0,5 = au hasard, 1,0 = parfait
score = roc_auc_score(y_test, proba)
print("Qualite du modele (AUC) :", round(score, 3))

Qualite du modele (AUC) : 0.757


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Étape 5.1 — On installe l'outil d'équité

In [ ]:
!pip install fairlearn -q
print("Outil d'equite installe.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.3 MB/s eta 0:00:00
Outil d'equite installe.


## Étape 5.2 — Taux d'acceptation par groupe d'âge

In [ ]:
from fairlearn.metrics import MetricFrame, selection_rate
# On recupere l'age des personnes du groupe de test (colonne "age", en minuscules)
age_test = data.loc[X_test.index, "age"]
# Deux groupes : "30 ans ou moins" et "Plus de 30 ans"
groupe = (age_test > 30).map({True: "Plus de 30 ans", False: "30 ans ou moins"})
# Decision du modele : pret accorde si proba de bon payeur >= 0,5
decision = (proba >= 0.5).astype(int)
mesure = MetricFrame(metrics=selection_rate,
                     y_true=y_test, y_pred=decision,
                     sensitive_features=groupe)
print("Taux d'acceptation par groupe :")
print(mesure.by_group)

Taux d'acceptation par groupe :
age
30 ans ou moins    0.671053
Plus de 30 ans     0.782258
Name: selection_rate, dtype: float64


## Étape 6 — Comprendre POURQUOI le modèle décide

In [ ]:
import pandas as pd
# On associe a chaque colonne son poids dans la decision du modele
importance = pd.Series(modele.coef_[0], index=X.columns)
print("Les 10 facteurs qui poussent vers l'ACCEPTATION :")
print(importance.sort_values(ascending=False).head(10))
print()
print("Les 10 facteurs qui poussent vers le REFUS :")
print(importance.sort_values().head(10))

Les 10 facteurs qui poussent vers l'ACCEPTATION :
status_no checking account                                1.618059
purpose_car (used)                                        1.329810
other_debtors_guarantor                                   1.229415
savings_... >= 1000 DM                                    1.011559
credit_history_critical account/other credits existing    0.998695
savings_unknown/no savings account                        0.887522
status_... >= 200 DM / salary for at least 1 year         0.829178
other_installment_plans_none                              0.729530
employment_duration_4 <= ... < 7 years                    0.597932
savings_100 <= ... < 500 DM                               0.447792
dtype: float64

Les 10 facteurs qui poussent vers le REFUS :
foreign_worker_yes                        -0.992772
purpose_retraining                        -0.739696
purpose_car (new)                         -0.730520
housing_rent                              -0.610362
property_un